# Sorax sur Colab T4 — corpus, entraînement, export, `.sb3`

> Projet **Snowoo-** · moteur **Sorax**

Ce notebook fait toute la chaîne, dans cet ordre :

1. monte Google Drive (les artefacts **survivent à la fermeture du navigateur**) ;
2. récupère le dépôt ;
3. construit la grammaire des blocs, le corpus et le tokenizer ;
4. entraîne Sorax **sur GPU** (`tools/train_torch.py`) ;
5. exporte les poids en **int8** (`sorax_core.bin`) ;
6. fabrique le projet **Scratch** `sorax-scratch.sb3` ;
7. vérifie la parité des trois moteurs (NumPy ↔ JS ↔ blocs Scratch).

**Astuce Colab** : *Exécution ▸ Tout exécuter*. Une coupure ? Relancez : le corpus,
le tokenizer et les checkpoints sont repris depuis Drive, et l'entraînement
repart du dernier checkpoint.

| Réglage | Où | Défaut |
|---|---|---|
| Palier | cellule « Paramètres » | `nano` (~148 k paramètres) |
| Taille du corpus | cellule « Paramètres » | `colab` (40 000 exemples) |
| Durée d'entraînement | `STEPS` | 4 000 pas (~25 min sur T4) |


In [ ]:
#@title Paramètres (modifiable à la main) { display-mode: "form" }
import os

PARAMS = {
    'PALIER': 'nano',          # pico | nano | mini | plus
    'CORPUS_PRESET': 'colab',  # demo (2 000) | colab (40 000) | maxi (120 000)
    'STEPS': '4000',           # pas d'entraînement GPU
    'BATCH_SIZE': '32',
    'REPO': '/content/incode',
    'BRANCHE': 'develop',
    'REPO_URL': 'https://github.com/Snowoo-2z/incode.git',
    'DRIVE': '/content/drive/MyDrive/sorax',
}

# publiés dans l'environnement : les cellules %%bash les voient telles quelles
os.environ.update(PARAMS)
PALIER = PARAMS['PALIER']

print(f"palier {PALIER} | corpus {PARAMS['CORPUS_PRESET']} | "
      f"{PARAMS['STEPS']} pas | lot {PARAMS['BATCH_SIZE']}")


## 1. Google Drive

Tout ce qui est écrit dans `DRIVE` est conservé entre les sessions.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

for sous in ('corpus', f'checkpoints/{PALIER}', f'export/{PALIER}', 'sb3'):
    os.makedirs(os.path.join(DRIVE, sous), exist_ok=True)
print('espace de travail :', DRIVE)


## 2. Le dépôt

Le notebook clone la branche demandée puis travaille dans `/content/incode`.
Relancer cette cellule après un `git pull` met à jour le code.


In [ ]:
%%bash
set -e
if [ -d "$REPO/.git" ]; then
    git -C "$REPO" fetch --depth 1 origin "$BRANCHE"
    git -C "$REPO" checkout -q "$BRANCHE"
    git -C "$REPO" pull --ff-only origin "$BRANCHE" || true
else
    git clone --depth 1 --branch "$BRANCHE" "$REPO_URL" "$REPO"
fi
cd "$REPO"
git log --oneline -1
ls sorax


## 3. Environnement

Colab fournit déjà PyTorch et Node. On vérifie juste que le GPU est bien là.


In [ ]:
import os, subprocess, sys

REPO = os.environ['REPO']
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, 'sorax'))

import torch
print('torch', torch.__version__, '| CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
else:
    print('⚠️  pas de GPU : activez-le (Exécution ▸ Modifier le type d\'exécution ▸ T4 GPU)')
print(subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip())


## 4. Grammaire, corpus, tokenizer

La grammaire des blocs est extraite du dépôt (elle doit rester la vérité du compilateur).
Le corpus et le tokenizer sont écrits **sur Drive** : une seule génération suffit.


In [ ]:
%%bash
cd "$REPO"
set -e

# 1) grammaire réelle des blocs de l'IDE
node sorax/tools/dump_grammar.mjs >/dev/null
cp -f sorax/assets/grammar.json "$DRIVE/grammar.json"

# 2) corpus (sauté s'il existe déjà)
if [ ! -f "$DRIVE/corpus/train.jsonl" ]; then
    python sorax/tools/make_corpus.py --preset "$CORPUS_PRESET" --out "$DRIVE/corpus"
else
    wc -l "$DRIVE/corpus/train.jsonl" "$DRIVE/corpus/val.jsonl"
fi

# 3) tokenizer (sauté s'il existe déjà)
if [ ! -f "$DRIVE/tokenizer.json" ]; then
    python sorax/tools/build_tokenizer.py --corpus "$DRIVE/corpus" --out "$DRIVE/tokenizer.json"
fi
cp -f "$DRIVE/tokenizer.json" sorax/assets/tokenizer.json


## 5. Entraînement GPU

`tools/train_torch.py` est le jumeau PyTorch du modèle NumPy : mêmes tenseurs, mêmes
hyper-paramètres. La perte ne porte que sur la **réponse** (la consigne n'est pas à
recopier). Le checkpoint `last.npz` est réécrit à chaque évaluation : relancer la
cellule reprend l'entraînement là où il s'est arrêté.

> Sur T4, comptez ~25 min pour 4 000 pas en `nano` sur le corpus `colab`.


In [ ]:
%%bash
cd "$REPO"
set -e
RESUME=""
if [ -f "$DRIVE/checkpoints/$PALIER/last.npz" ]; then RESUME="--resume $DRIVE/checkpoints/$PALIER/last.npz"; fi
python sorax/tools/train_torch.py \
  --config "$PALIER" \
  --corpus "$DRIVE/corpus" \
  --tokenizer "$DRIVE/tokenizer.json" \
  --out "$DRIVE/checkpoints/$PALIER" \
  --steps "$STEPS" --batch-size "$BATCH_SIZE" \
  $RESUME


In [ ]:
import json, os

chemin = f'{DRIVE}/checkpoints/{PALIER}/rapport_entrainement.json'
if os.path.exists(chemin):
    rapport = json.load(open(chemin, encoding='utf-8'))
    print('pas :', rapport['steps'], '| durée :', rapport['duree_min'], 'min',
          '| appareil :', rapport.get('appareil'))
    print('meilleure val_loss :', rapport['best_val_loss'])
    dernier = rapport['history'][-1] if rapport['history'] else {}
    print('dernier point :', dernier)


## 6. Export int8 + projet Scratch

L'export quantifie chaque ligne de poids (un octet par poids + une échelle `float32`),
puis `tools/build_sb3.mjs` transforme le dump en **blocs Scratch** : listes de poids,
procédures warp, cache K/V, tokenizer en tables.


In [ ]:
%%bash
cd "$REPO"
set -e
python sorax/tools/export.py \
  --config "$PALIER" \
  --ckpt "$DRIVE/checkpoints/$PALIER/last.npz" \
  --corpus "$DRIVE/corpus" --tokenizer "$DRIVE/tokenizer.json" \
  --out "$DRIVE/export/$PALIER"

node sorax/tools/build_sb3.mjs \
  --bin "$DRIVE/export/$PALIER/sorax_core.bin" \
  --out "sorax/assets/export/$PALIER/sorax-scratch.sb3" \
  --context 128 --max-new 48

mkdir -p "static/sorax"
cp -f "sorax/assets/export/$PALIER/sorax-scratch.sb3" "$DRIVE/sb3/sorax-scratch-$PALIER.sb3"
cp -f "sorax/assets/export/$PALIER/sorax-scratch.sb3" "static/sorax/sorax-scratch.sb3"
ls -la "$DRIVE/export/$PALIER" "$DRIVE/sb3"


## 7. Vérifications

Trois moteurs doivent dire **exactement la même chose** : NumPy (entraînement),
JavaScript (navigateur) et les blocs Scratch (le `.sb3`).


In [ ]:
%%bash
cd "$REPO"
set -e
echo "── structure du .sb3"
node sorax/tools/validate_sb3.mjs || true
echo "── parité NumPy ↔ JS"
python sorax/tools/check_runtime_parity.py "$DRIVE/export/$PALIER/sorax_core.bin" || true
echo "── parité blocs Scratch ↔ JS"
node sorax/tools/test_scratch_engine.mjs \
  --sb3 "sorax/assets/export/$PALIER/sorax-scratch.sb3" \
  --bin "$DRIVE/export/$PALIER/sorax_core.bin" \
  --prompt "abc" --tokens 3 || true
echo "── décodeur, jeton par jeton"
node sorax/tools/test_decode.mjs \
  --sb3 "sorax/assets/export/$PALIER/sorax-scratch.sb3" \
  --bin "$DRIVE/export/$PALIER/sorax_core.bin" || true


## 8. Sorax répond

On déroule le projet Scratch pour de vrai (drapeau vert → consigne → génération) :
c'est le même code qui tournera dans le navigateur du visiteur.


In [ ]:
%%bash
cd "$REPO"
for demande in "Cree un jeu de Pong avec un score" \
                "Un personnage qui saute sur des plateformes" \
                "Ajoute un chronometre de 30 secondes"; do
  echo "───────────────────────────────────────"
  node sorax/tools/run_sb3.mjs \
    --sb3 "sorax/assets/export/$PALIER/sorax-scratch.sb3" \
    "$demande" | tail -n 12
done


## 9. Récupérer les fichiers

Tout est déjà sur Drive (`MyDrive/sorax/`). Cette cellule fait aussi un `.zip`
pratique à télécharger/ranger, et prévient que le `.sb3` peut être ouvert
directement dans Scratch ou TurboWarp.


In [ ]:
import os, shutil, zipfile

archive = f'{DRIVE}/sb3/sorax-{PALIER}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(f'{DRIVE}/export/{PALIER}/sorax_core.bin', f'sorax_core.bin')
    z.write(f'{DRIVE}/export/{PALIER}/sorax_meta.json', f'sorax_meta.json')
    z.write(f'{DRIVE}/sb3/sorax-scratch-{PALIER}.sb3', f'sorax-scratch-{PALIER}.sb3')
    z.write(f'{DRIVE}/tokenizer.json', 'tokenizer.json')
    for f in ('best.npz', 'last.npz', 'rapport_entrainement.json'):
        p = f'{DRIVE}/checkpoints/{PALIER}/{f}'
        if os.path.exists(p):
            z.write(p, f'checkpoints/{f}')
print('archive :', archive, f'({os.path.getsize(archive) / 1e6:.1f} Mo)')
print()
print('Projet Scratch  :', f'{DRIVE}/sb3/sorax-scratch-{PALIER}.sb3')
print('Poids int8      :', f'{DRIVE}/export/{PALIER}/sorax_core.bin')
print()
print('Ouvrez le .sb3 dans TurboWarp (ou Scratch) : cliquez sur le drapeau vert,')
print('cliquez sur Sorax, tapez votre demande — le réseau s\'exécute en blocs.')


## Ensuite

- **Reprendre l'entraînement** : augmentez `STEPS`, relancez la cellule 5 (reprise
  automatique depuis `last.npz`).
- **Passer au palier au-dessus** : changez `PALIER` (`mini` ≈ 5 M paramètres) ;
  le corpus/tokenizer restent valables, seul le contexte change.
- **Vérifier après chaque changement** : rejouez la cellule 7 (parités) — elle doit
  afficher « Le projet Scratch calcule exactement la même chose que le runtime. »
